In [ ]:
# --- Colab bootstrap: in Colab, clone + install the repo; locally, do nothing ---
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/CKmellow/per3-genotype-clock-gene-timing.git"
BRANCH = "main"  # change to your branch name to test it in Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content/per3-genotype-clock-gene-timing")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(repo)], check=True)
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    src = str(repo / "src")
else:  # local: find the repo root (works from notebooks/ or notebooks/stages/)
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)
    src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)  # makes `import per3clock` work without a kernel restart

from per3clock import config  # noqa: E402

config.set_seeds()
config.ensure_dirs()
print("per3clock root:", config.ROOT, "| Colab:", IN_COLAB)

# Stage 2 · Data acquisition

All data come from GEO series **GSE39445** (platform **GPL15331**, Agilent-026817 University_Surrey_HumanSleep_44k_v1). Small source files are committed in `data/raw/`. The ~100 MB whole-transcriptome series matrix is downloaded on first use and cached in `data/external/`. Everything is loaded through `per3clock.data`, the same functions used by every later stage. The full data dictionary is in `data/README.md`, and the verification report is `results/tables/data_verification.md`.

| File | What it is |
|---|---|
| `data/raw/GSE39445_sample_metadata.csv` | one row per sample: subject, genotype, condition, time awake, clock time, circadian phase |
| `data/raw/GSE39445_clock_genes.csv` | the same samples × 14 probes for 8 clock genes (log2 expression) |
| `data/raw/GPL15331_probe_annotation.csv` | probe ID → gene symbol, name, Ensembl ID, location |
| `data/raw/GSE39445.txt` | GEO SOFT sample headers (provenance) |
| `data/external/GSE39445_series_matrix.txt.gz` | whole transcriptome: 43,758 probes × 438 samples (downloaded) |

In [ ]:
import pandas as pd
from IPython.display import display

from per3clock import data

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## 2.1 Sample metadata

`load_metadata()` reads the metadata CSV, recovers the 11 samples whose subject and genotype were blank (from the sample title and the subject's other samples), parses `clock_time` into decimal hours, and adds `condition_code`, `timepoint` and `is_recovered`.

In [ ]:
meta = data.load_metadata()
print(meta.shape)
display(meta.head())
display(meta.dtypes.to_frame("dtype").T)

| Column | Meaning | Units / values |
|---|---|---|
| `sample_id` | GEO sample accession | GSM… |
| `title` | GEO sample title | `Subject_<id>_<SE/SR>_T<n>` |
| `subject` | participant ID | 26 IDs (AF0004…AF0335) |
| `per3_genotype` | PER3 VNTR genotype | `4/4`, `5/5` |
| `sleep_condition` | prior week of sleep | `Sleep Extension` (sufficient, control), `Sleep Restriction` (6 h/night) |
| `condition_code` | short code | `SE`, `SR` |
| `timepoint`, `timepoint_n` | sample number within the 40-h wake period | T1–T10 / 1–10 |
| `hours_awake` | time since waking | h (7.5, 10.5, …, 34.5) |
| `clock_time` | clock time of blood draw | `H:MM` (no date) |
| `clock_time_h` | same, decimal | h in [0, 24) |
| `circadian_phase_h` | time relative to the participant's plasma-melatonin rhythm (GEO) | h, 3-h steps from −15 to 15 |
| `is_recovered` | subject/genotype recovered from title | bool |

In [ ]:
display(meta.loc[meta.is_recovered, ["sample_id", "title", "subject", "per3_genotype", "condition_code", "timepoint"]])

## 2.2 Clock-gene expression

`load_clock_genes()` returns a long table (sample × gene). **Default: one probe per gene**, the probe with the highest mean expression across all 438 samples (`select_probes`, `collapse="max_mean"`). Averaging would mix in near-background probes. For example, two of PER3's three probes have mean log2 3.9 and 4.2. The rule uses only each probe's overall mean, never time, rhythmicity or genotype, so it cannot bias the later tests. `collapse="mean"` (average probes) and `collapse=None` (probe level) remain available.

In [ ]:
clock = data.load_clock_genes()                  # default: selected probe per gene
probes = data.load_clock_genes(collapse=None)    # every probe
print("long table:", clock.shape, "| probe-level:", probes.shape)
display(clock[["sample_id", "gene", "probe_id", "expression", "n_probes", "subject", "per3_genotype", "condition_code", "timepoint", "clock_time_h"]].head(8))

selection = data.clock_probe_selection()         # also saved as results/tables/selected_probes.csv
display(selection)

`expression` is the GEO quantile-normalised **log2** signal (unitless). Each gene has 1–3 Agilent probes, and their IDs match the GPL15331 annotation and the series matrix. The selected probes are PER2 16821, PER3 25341, NR1D1 33493, CRY1 22736 and NR1D2 38683; CLOCK, ARNTL and PER1 have a single probe. PER3 remains the weakest primary gene (mean log2 5.5 against an array median of 6.7).

## 2.3 Probe annotation (GPL15331)

In [ ]:
annot = data.load_probe_annotation()
print(annot.shape, "| probes with a gene symbol:", annot.GENE_SYMBOL.notna().sum())
display(annot.head(3))
display(annot.dropna(subset=["GENE_SYMBOL"]).head(3))

| Column | Meaning |
|---|---|
| `ID` (index) | integer probe ID (= series-matrix `ID_REF`) |
| `PROBE_NAME` | Agilent probe name, or a control name (e.g. `GE_BrightCorner`) |
| `GENE_SYMBOL`, `GENE_NAME` | HGNC symbol and name (blank for controls) |
| `ENSEMBL_ID` | Ensembl identifier |
| `CHROMOSOMAL_LOCATION` | probe coordinates |

## 2.4 GEO SOFT sample headers (provenance)

In [ ]:
soft = data.load_soft_samples()
print(soft.shape)
display(soft.head(3))
print("Samples storing genotype under 'genotype' instead of 'genotype/variation':", soft["genotype"].notna().sum())

## 2.5 Whole-transcriptome series matrix

Downloaded once (~100 MB, about 30 s in Colab) to `data/external/` and cached. `load_expression_matrix()` checks that there are 438 samples and a plausible number of probes. `collapse_probes_to_genes()` makes a genes × samples matrix, keeping by default the same outcome-blind choice of one probe per gene (highest mean across all samples).

In [ ]:
expr, chars = data.load_expression_matrix(with_characteristics=True)
print("probes x samples:", expr.shape)
print(f"log2 range {expr.values.min():.2f}-{expr.values.max():.2f}, median {pd.Series(expr.values.ravel()).median():.2f}, NaN: {int(expr.isna().sum().sum())}")
genes = data.collapse_probes_to_genes(expr, annot)
print("genes x samples:", genes.shape)
display(chars.head(3))

In [ ]:
# Check: the clock-gene CSV equals the series matrix for all 14 probes x 438 samples
wide = probes.pivot(index="sample_id", columns="probe_id", values="expression")
diff = (expr.loc[wide.columns, wide.index].T - wide).abs().to_numpy().max()
print("max |clock CSV - series matrix| =", diff)
assert diff < 1e-6

## 2.6 Study summary and the shared analysis sets

`analysis_samples()` defines the sample set every later stage uses (issue #10):
- **default**: Sleep Extension, subjects with ≥6 of 10 timepoints (main analysis);
- `strict=True`: the original authors' rule (drop series with 2 consecutive or >2 missing timepoints), as a sensitivity analysis;
- `for_phase_analysis=True`: also drops AF0182 in SE, whose melatonin-phase labels are inconsistent. Use it whenever time is `circadian_phase_h`. AF0182 stays in clock-time analyses because its clock times are real.

In [ ]:
summary = (
    meta.groupby(["per3_genotype", "sleep_condition"])
    .agg(subjects=("subject", "nunique"), samples=("sample_id", "size"))
    .unstack("sleep_condition")
)
display(summary)

tp = meta.pivot_table(index=["per3_genotype", "subject"], columns="condition_code", values="sample_id", aggfunc="count", fill_value=0)
display(tp)

variants = {
    "default (main)": dict(),
    "strict=True (sensitivity)": dict(strict=True),
    "for_phase_analysis=True": dict(for_phase_analysis=True),
    "Sleep Restriction (sensitivity)": dict(condition="Sleep Restriction"),
}
rows = []
for label, kw in variants.items():
    s, excl = data.analysis_samples(**kw, metadata=meta, return_exclusions=True)
    gt = s.drop_duplicates("subject").per3_genotype.value_counts()
    rows.append({"set": label, "subjects": s.subject.nunique(), "4/4": gt.get("4/4", 0),
                 "5/5": gt.get("5/5", 0), "samples": len(s), "excluded": ", ".join(excl.subject)})
display(pd.DataFrame(rows))

_, excl = data.analysis_samples(strict=True, for_phase_analysis=True, metadata=meta, return_exclusions=True)
print("Why subjects are excluded (Sleep Extension, strict + phase):")
display(excl)

### Interpretation

- **Size and design.** 438 blood samples from 26 participants: 12 PER3 4/4 and 14 PER3 5/5, all homozygous, so the comparison is a clean two-group contrast. 221 samples are from the sufficient-sleep (SE) condition and 217 from sleep restriction (SR). Samples were taken every 3 h from 7.5 to 34.5 h awake, which covers more than one full 24-h cycle.
- **Data integrity.** After recovery, every sample has a subject and genotype. The 11 recovered genotypes agree with GEO's own (differently named) genotype field. The clock-gene CSV is identical to the series matrix (max difference 0). The metadata agree with GEO on every field.
- **Shared analysis sets.** Main analysis: 24 participants (11 × 4/4, 13 × 5/5), 221 SE samples. Strict sensitivity set: 20 participants (9 × 4/4, 11 × 5/5), 193 samples. Melatonin-phase analyses: 23 participants (AF0182 dropped), 214 samples. AF0079 and AF0091 have only SR samples, and AF0186 and AF0295 have only SE samples. The genotype groups are small, so later stages should report effect sizes with confidence intervals, not only p-values.
- **One probe per gene.** Using the most highly expressed probe avoids diluting signals with near-background probes (two of PER3's three). The choice is outcome-blind.
- **Things later stages must handle** (details in `results/tables/data_verification.md`):
  1. AF0182 SE T4/T5 have identical circadian phase with irregular spacing, probably an annotation error. AF0182 is excluded from phase analyses only.
  2. PER3 is weakly expressed on this array, so Part 2 should confirm it is above background.
  3. `circadian_phase_h` is binned in 3-h steps. The implied phase-0 clock time averages ~04:10, close to the published melatonin midpoint, but the exact definition is unconfirmed.
- The whole-transcriptome matrix (43,758 probes, ~19.5k genes) is available for the genome-wide rhythmicity scan, the enrichment analysis and the ML stage.

**Can these data answer the question?** Yes, with caveats. The data have everything the question needs: PER3-genotyped participants in two homozygous groups, repeated blood samples spanning more than 24 h (so a peak time can be estimated per person), the three target genes measured on the array, and a melatonin-based phase for objective 3. The limits are a small sample (~12 per genotype, so only fairly large shifts are detectable), 3-hourly sampling and 3-h phase bins (coarse timing), and a single cohort with no independent replication set. These limit precision and power, not validity, and stage 8 must state them.